# Script Universal de Analise de Dados (Pandas + Plotly) para API

## Carregando Bibliotecas

In [12]:
import pandas as pd
import plotly.express as px
import requests as req

## Carregando Base de Dados (Pandas)

In [39]:
# Link da Documentação da API: https://www.alphavantage.co/documentation/

url = "https://www.alphavantage.co/query?function=TIME_SERIES_MONTHLY&symbol=IBM&apikey=demo"
df = req.get(url).json()
df = pd.DataFrame(df['Monthly Time Series']).T

In [40]:
float_columns = ['1. open', '2. high', '3. low', '4. close', '5. volume']
df[float_columns] = df[float_columns].astype(float).round(2)

## Dignóstico Rápido

In [41]:
print("--- PRIMEIRAS LINHAS ---")
display(df.head())

print("\n--- INFORMAÇÕES E TIPOS DE DADOS ---")
print(df.info())

print("\n--- RESUMO ESTATÍSTICO ---")
display(df.describe())


--- PRIMEIRAS LINHAS ---


,1. open,2. high,3. low,4. close,5. volume
2026-10-01,230.76,234.27,224.50,225.62,8752093.0
2026-09-30,231.00,251.82,218.60,219.93,114235848.0
2026-08-31,228.33,241.80,223.30,233.87,111411516.0
2026-07-31,279.66,311.80,199.19,223.65,299532613.0
2026-06-30,322.55,332.46,243.68,281.21,220624784.0



--- INFORMAÇÕES E TIPOS DE DADOS ---
<class 'pandas.DataFrame'>
Index: 323 entries, 2026-10-01 to 1999-12-31
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   1. open    323 non-null    float64
 1   2. high    323 non-null    float64
 2   3. low     323 non-null    float64
 3   4. close   323 non-null    float64
 4   5. volume  323 non-null    float64
dtypes: float64(5)
memory usage: 15.1+ KB
None

--- RESUMO ESTATÍSTICO ---


,1. open,2. high,3. low,4. close,5. volume
count,323.000000,323.000000,323.000000,323.000000,3.230000e+02
mean,141.078328,148.802539,133.533560,141.234241,1.229094e+08
std,49.155867,52.226613,45.686482,49.054748,4.644506e+07
min,59.200000,74.200000,54.010000,58.310000,8.752093e+06
25%,105.370000,112.905000,98.980000,105.230000,8.874619e+07
50%,133.420000,139.470000,126.000000,133.570000,1.114115e+08
75%,164.970000,173.660000,159.315000,166.095000,1.459727e+08
max,322.550000,332.460000,295.700000,308.580000,3.149720e+08


## Limpeza e Tratamento de Dados

In [42]:
if df.isnull().values.any():
    print("\n--- VALORES NULOS ---")
    display(df.isnull().sum())
    print("--- REMOVENDO LINHAS COM VALORES NULOS ---")
    df_cleaned = df.dropna()
    display(df_cleaned.describe())

    if df_cleaned.duplicated().any():
        print("\n--- EXISTEM LINHAS DUPLICADAS ---")
        print(df_cleaned[df_cleaned.duplicated()])
        print("\n--- REMOVENDO LINHAS DUPLICADAS ---")
        df_cleaned = df_cleaned.drop_duplicates()
        display(df_cleaned.describe())


## Respondendo Perguntas do Dataframe

In [46]:
#Periodo de tempo disponível na API
inicio = df.index.min()
fim = df.index.max()
display(f"Período de tempo disponível na API: {inicio} a {fim}")

'Período de tempo disponível na API: 1999-12-31 a 2026-10-01'

In [48]:
#Quantidade de meses disponíveis na API
quantidade_meses = df.shape[0]
display(f"Quantidade de meses disponíveis na API: {quantidade_meses} meses")

'Quantidade de meses disponíveis na API: 323 meses'

In [50]:
#Mês de máximama e mínima
max = df['2. high'].max()
min = df['3. low'].min()
day_max = df.loc[df['2. high'] == max].index[0]
day_min = df.loc[df['3. low'] == min].index[0]

display(f"Mês com maior valor máximo: {day_max} com valor de: {max}")
display(f"Mês com menor valor mínimo: {day_min} com valor de: {min}")

'Mês com maior valor máximo: 2026-06-30 com valor de: 332.46'

'Mês com menor valor mínimo: 2002-10-31 com valor de: 54.01'

In [51]:
#Valor médio de abertura e fechamento
mean_open = df['1. open'].mean()
mean_close = df['4. close'].mean()
display(f"Valor médio de abertura: {mean_open:.2f}")
display(f"Valor médio de fechamento: {mean_close:.2f}")

'Valor médio de abertura: 141.08'

'Valor médio de fechamento: 141.23'

## Representacoes Graficas Rapidas

In [101]:
# Gráfico de Evolução dos Preços de Abertura e Fechamento ao longo do tempo
df_plot = df.sort_index()
df_plot = df_plot.rename(columns={'1. open': 'Preço de Abertura','4. close': 'Preço de Fechamento'})

fig = px.line(
    df_plot,
    x=df_plot.index,
    y=['Preço de Abertura', 'Preço de Fechamento'],
    title='<b>Evolução dos Preços de Abertura e Fechamento</b>',
    template='plotly_white',
    labels={
        'index': 'Mês/Ano',
        'value': 'Preço',
        'variable': 'Indicador',
    }
)

fig.show()


## Gráficos mais Avançados

### Regreção Linear com Máximas e Mínimas

In [105]:
## previsão de preços futuros usando regressão linear para os próximos 12 meses
from sklearn.linear_model import LinearRegression

reg = LinearRegression()
reg.fit(df_plot['Preço de Abertura'].values.reshape(-1, 1), df_plot['Preço de Fechamento'].values)
regression_line = reg.predict(df_plot['Preço de Abertura'].values.reshape(-1, 1))

In [107]:
regression_df = pd.DataFrame({
    'Preço de Abertura': df_plot['Preço de Abertura'],
    'Preço de Fechamento': df_plot['Preço de Fechamento'],
    'Linha de Regressão': regression_line
})

display(regression_df)

,Preço de Abertura,Preço de Fechamento,Linha de Regressão
1999-12-31,102.56,107.87,103.882963
2000-01-31,112.44,112.25,113.463614
2000-02-29,112.37,102.75,113.395735
2000-03-31,102.00,118.37,103.339931
2000-04-28,120.00,111.50,120.794556
...,...,...,...
2026-06-30,322.55,281.21,317.207582
2026-07-31,279.66,223.65,275.617087
2026-08-31,228.33,233.87,225.842313
2026-09-30,231.00,219.93,228.431415


In [108]:
#Grafico de Evolução dos Preços de Abertura e Fechamento ao longo do tempo com linha de regressão
fig = px.line(
    regression_df,
    x=regression_df.index,
    y=['Preço de Abertura', 'Preço de Fechamento', 'Linha de Regressão'],
    title='<b>Evolução dos Preços de Abertura e Fechamento com Linha de Regressão</b>',
    template='plotly_white',
    labels={
        'Mês/Ano': 'Mês/Ano',
        'value': 'Preço',
        'variable': 'Indicador',
        'Preço de Abertura': 'Preço de Abertura',
        'Preço de Fechamento': 'Preço de Fechamento',
        'Linha de Regressão': 'Linha de Regressão'
    }
)
fig.show()